In [1]:
import os
import tensorflow as tf

TRAIN_DIR = "../data/processed_top20/train"
VAL_DIR = "../data/processed_top20/val"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
SEED = 6767

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

TensorFlow version: 2.21.0
GPU available: False


In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

Found 10958 files belonging to 20 classes.
Found 1364 files belonging to 20 classes.
Classes: ['15068', '2357', '3001', '3002', '3003_6223', '3004_3065', '3007', '3009', '3010', '3034', '3069b', '3710', '3832', '60484', '60592_60601', '6232', '85080', '85984', '87079', '92013']
Number of classes: 20


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [4]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1.0 / 255.0),

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ rescaling (Rescaling)                │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 224, 224, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 112, 112, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 56, 56, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 20)                  │           5,140 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 393,556 (1.50 MB)

 Trainable params: 393,556 (1.50 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [6]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 641s 2s/step - accuracy: 0.2280 - loss: 2.3929 - val_accuracy: 0.4626 - val_loss: 1.6604
Epoch 2/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 712s 2s/step - accuracy: 0.4731 - loss: 1.5672 - val_accuracy: 0.5630 - val_loss: 1.3108
Epoch 3/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 511s 1s/step - accuracy: 0.5597 - loss: 1.3174 - val_accuracy: 0.6012 - val_loss: 1.1862
Epoch 4/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 532s 1s/step - accuracy: 0.5972 - loss: 1.1866 - val_accuracy: 0.6408 - val_loss: 1.0707
Epoch 5/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 469s 1s/step - accuracy: 0.6387 - loss: 1.0738 - val_accuracy: 0.6723 - val_loss: 1.0195
Epoch 6/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 465s 1s/step - accuracy: 0.6566 - loss: 1.0167 - val_accuracy: 0.6906 - val_loss: 0.9886
Epoch 7/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 499s 1s/step - accuracy: 0.6808 - loss: 0.9445 - val_accuracy: 0.7111 - val_loss: 0.8921
Epoch 8/20
343/343 ━━━━━━━━━━━━━━━━━━━━ 808s 2s/step - accuracy: 0.7071 - loss: 0.8649 - val_accu

In [7]:
os.makedirs("../models", exist_ok=True)
model.save("../models/cnn_from_scratch.keras")
print("Model saved to ../models/cnn_from_scratch.keras")

Model saved to ../models/cnn_from_scratch.keras
